# Multipose2 Multimodal Adapter Training

This notebook validates the new Multipose2 input-adapter architecture on multichannel data. It is intentionally separate from `train_Multipose2.ipynb`, which remains the basic tutorial notebook.

Use this notebook to train comparable runs with:

- `adapter_type="linear"`: the default 1x1 channel-fusion adapter.
- `adapter_type="msca_lite"`: the lightweight MSCA-inspired multiscale adapter.

Keep the same data split and training settings for both adapters so the comparison is meaningful.

In [ ]:
from pathlib import Path
import sys

# If running from the repo, prefer the local checkout so notebook validation uses your current code.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "multipose2").exists() else NOTEBOOK_DIR.parent
if (REPO_ROOT / "multipose2").exists():
    sys.path.insert(0, str(REPO_ROOT.resolve()))
    print(f"Using local multipose2 repo: {REPO_ROOT.resolve()}")
else:
    print("Local repo not found. In Colab, install with:")
    print("!pip install git+https://github.com/kevins-winter/multipose2.git")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from multipose2 import core, io, metrics, models, train, transforms

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
io.logger_setup()

use_gpu = core.use_gpu()
print(f"GPU available: {use_gpu}")

## 1. Configure Data And Experiment

Set `train_dir`, `test_dir`, and `mask_filter` for your data. For channel axes:

- Use `channel_axis=-1` for `H x W x C` images.
- Use `channel_axis=0` for `C x H x W` images.
- Use `channel_axis=None` only if you want Multipose2 to infer the channel axis.

In [ ]:
# Required: update this path.
train_dir = Path("/content/gdrive/MyDrive/PATH-TO-FILES/")

# Optional: set to None if you do not have a separate test folder.
test_dir = None

# Common values: "_seg.npy" for Multipose2/Cellpose GUI labels, or "_masks.tif" for TIFF masks.
mask_filter = "_seg.npy"

# Set this explicitly for multichannel validation when possible.
channel_axis = -1

# Training configuration.
pretrained_model = "cpsam"
model_name_prefix = "multipose2_multimodal"
adapter_types = ["linear", "msca_lite"]

n_epochs = 100
learning_rate = 1e-5
weight_decay = 0.1
batch_size = 1
nimg_per_epoch = None  # None uses all training images each epoch.

if not train_dir.exists():
    raise FileNotFoundError(f"train_dir does not exist: {train_dir}")
if test_dir is not None and not Path(test_dir).exists():
    raise FileNotFoundError(f"test_dir does not exist: {test_dir}")

## 2. Load Data And Infer Channel Count

This cell loads image/mask pairs and verifies the inferred channel count before model creation. The model should be initialized with the real channel count so training does not start with the wrong adapter shape.

In [ ]:
output = io.load_train_test_data(
    str(train_dir),
    None if test_dir is None else str(test_dir),
    mask_filter=mask_filter,
)
train_data, train_labels, train_files, test_data, test_labels, test_files = output

print(f"train images: {len(train_data)}")
print(f"test images: {0 if test_data is None else len(test_data)}")

def infer_nchan_from_image(img, channel_axis=None):
    converted = transforms.convert_image(img, channel_axis=channel_axis, do_3D=False)
    return converted.shape[-1], converted.shape

nchan, converted_shape = infer_nchan_from_image(train_data[0], channel_axis=channel_axis)
print(f"first raw image shape: {train_data[0].shape}")
print(f"converted image shape: {converted_shape}")
print(f"inferred nchan: {nchan}")

if nchan < 1:
    raise ValueError("Could not infer a valid channel count")

## 3. Visualize Channels

For multichannel data, inspect individual channels instead of forcing the image into RGB. This helps catch wrong `channel_axis` settings before training.

In [ ]:
img0 = transforms.convert_image(train_data[0], channel_axis=channel_axis, do_3D=False)
channels_to_show = list(range(min(nchan, 8)))

fig, axes = plt.subplots(1, len(channels_to_show), figsize=(3 * len(channels_to_show), 3), dpi=120)
if len(channels_to_show) == 1:
    axes = [axes]
for ax, c in zip(axes, channels_to_show):
    ax.imshow(img0[..., c], cmap="gray")
    ax.set_title(f"channel {c}")
    ax.axis("off")
plt.tight_layout()

## 4. Train Adapter Runs

This trains one model per adapter type. Start with `adapter_types = ["linear"]` for a plumbing check, then run both adapters for comparison.

In [ ]:
results = {}

for adapter_type in adapter_types:
    print("=" * 80)
    print(f"Training adapter_type={adapter_type}, nchan={nchan}")

    model = models.CellposeModel(
        gpu=use_gpu,
        pretrained_model=pretrained_model,
        nchan=nchan,
        adapter_type=adapter_type,
    )

    model_name = f"{model_name_prefix}_{adapter_type}_{nchan}ch"
    run_nimg_per_epoch = len(train_data) if nimg_per_epoch is None else nimg_per_epoch

    model_path, train_losses, test_losses = train.train_seg(
        model.net,
        train_data=train_data,
        train_labels=train_labels,
        test_data=test_data,
        test_labels=test_labels,
        channel_axis=channel_axis,
        batch_size=batch_size,
        n_epochs=n_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        nimg_per_epoch=run_nimg_per_epoch,
        model_name=model_name,
        save_path=str(train_dir),
    )

    results[adapter_type] = {
        "model_path": model_path,
        "train_losses": train_losses,
        "test_losses": test_losses,
    }
    print(f"Saved {adapter_type} model to: {model_path}")

## 5. Compare Training Curves

In [ ]:
plt.figure(figsize=(8, 5), dpi=120)
for adapter_type, run in results.items():
    plt.plot(run["train_losses"], label=f"{adapter_type} train")
    if run["test_losses"] is not None and np.any(run["test_losses"]):
        plt.plot(run["test_losses"], linestyle="--", label=f"{adapter_type} test")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()

## 6. Evaluate On Held-Out Data

This runs each trained model on `test_data` and reports average precision. Skip this section if you do not have a test set.

In [ ]:
eval_results = {}

if test_data is None or test_labels is None:
    print("No test data loaded; skipping held-out evaluation.")
else:
    for adapter_type, run in results.items():
        print("=" * 80)
        print(f"Evaluating adapter_type={adapter_type}")
        model = models.CellposeModel(
            gpu=use_gpu,
            pretrained_model=run["model_path"],
            nchan=nchan,
            adapter_type=adapter_type,
        )
        masks = model.eval(test_data, batch_size=32, channel_axis=channel_axis)[0]
        ap = metrics.average_precision(test_labels, masks)[0]
        eval_results[adapter_type] = {"masks": masks, "ap": ap}
        print(f"mean AP @ IoU 0.50: {ap[:, 0].mean():.3f}")

## 7. Inspect Predictions

In [ ]:
if not eval_results:
    print("No eval results to plot.")
else:
    max_images = min(4, len(test_data))
    adapter_names = list(eval_results)
    nrows = 2 + len(adapter_names)
    fig, axes = plt.subplots(nrows, max_images, figsize=(3 * max_images, 3 * nrows), dpi=120)
    if max_images == 1:
        axes = axes[:, None]

    for k in range(max_images):
        img = transforms.convert_image(test_data[k], channel_axis=channel_axis, do_3D=False)
        axes[0, k].imshow(img[..., 0], cmap="gray")
        axes[0, k].set_title("image channel 0")
        axes[1, k].imshow(test_labels[k][0] if test_labels[k].ndim == 3 else test_labels[k], cmap="magma")
        axes[1, k].set_title("ground truth")
        for row, adapter_type in enumerate(adapter_names, start=2):
            axes[row, k].imshow(eval_results[adapter_type]["masks"][k], cmap="magma")
            axes[row, k].set_title(adapter_type)

    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()

## Notes For Interpreting Results

- If `linear` fails, debug data shape/channel plumbing first.
- If `linear` works but `msca_lite` fails, keep `linear` as the baseline and tune the richer adapter later.
- If `msca_lite` improves held-out metrics without obvious overfitting or memory problems, it is a candidate default for multimodal experiments, but not yet for general use.